# EEG baseline figures

Validation-only comparisons for the five fixed architectures. Locked test results should be added only after the model-selection protocol is finalized.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULT_ROOT = Path('../data/seed-0/benchmarking')
result_files = sorted(RESULT_ROOT.glob('*/*/baseline_results.jsonl'))
result_files

In [ ]:
records = []
for path in result_files:
    with path.open() as file:
        for line in file:
            entry = json.loads(line)
            records.append({
                'dataset': entry['dataset'],
                'baseline': entry['baseline'],
                'balanced_accuracy': entry['validation']['balanced_accuracy'],
                'accuracy': entry['validation']['accuracy'],
                'macro_f1': entry['validation']['macro_f1'],
                'parameters': entry['parameter_count'],
                'latency_ms': entry['validation']['inference_ms_per_trial'],
                'training_seconds': sum(fold['training_seconds'] for fold in entry['fold_results']),
                'best_epoch': sum(fold['best_epoch'] for fold in entry['fold_results']) / len(entry['fold_results']),
            })
results = pd.DataFrame(records)
results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4), constrained_layout=True)
for dataset, group in results.groupby('dataset'):
    axes[0].plot(group['baseline'], group['balanced_accuracy'], marker='o', label=dataset)
    axes[1].scatter(group['parameters'], group['balanced_accuracy'], label=dataset)
    axes[2].scatter(group['latency_ms'], group['balanced_accuracy'], label=dataset)
axes[0].set_ylabel('Validation balanced accuracy')
axes[0].tick_params(axis='x', rotation=45)
axes[1].set(xlabel='Trainable parameters', ylabel='Validation balanced accuracy')
axes[2].set(xlabel='Inference ms / trial', ylabel='Validation balanced accuracy')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()

In [ ]:
metric_table = results.pivot_table(
    index='baseline',
    columns='dataset',
    values=['balanced_accuracy', 'macro_f1', 'training_seconds'],
)
metric_table